After we uploaded our map product files into Google Earth Engine, GEE converted some of our field polygons into GeometryCollections, which can contain both polygons and linestrings. The linestrings are degenerate parts of the shape resulting from the polygon simplification + erosion step of the inference procedure. We need to re-download our map product and remove the linestrings.

In [4]:
import numpy as np
np.bool = np.bool_ 
import pandas as pd
import os
from tqdm import tqdm

import mxnet as mx
from mxnet import gluon
from mxnet import autograd
from mxnet import image

import sys

from sklearn.metrics import matthews_corrcoef

import random

import rasterio
from rasterio.enums import ColorInterp
from rasterio.features import shapes
from scipy import ndimage

import matplotlib.pyplot as plt
%matplotlib inline

import higra as hg
import torch
import pickle

import itertools

import imageio.v3 as imageio
from skimage.color import label2rgb
import cv2
from skimage.segmentation import mark_boundaries
from skimage.segmentation import find_boundaries

from shapely.geometry import shape
import shapely
from shapely import Polygon, MultiPolygon, orient_polygons
from shapely.wkt import loads
from shapely import ops
from shapely.ops import polygonize

import geopandas as gpd
import pyproj

import ast

import glob

import multiprocessing

import requests

# Download the map product files we uploaded to Google Earth Engine

In [2]:
import ee

ee.Authenticate()
ee.Initialize(project='ee-kerrilu')

In [5]:
fc = ee.FeatureCollection('projects/kerrilu/assets/US-fields/FieldSetsUS_1')
url = fc.getDownloadURL(**{
  'filetype': 'CSV',
})
r = requests.get(url, stream=True)
with open('inference_data/outputs/FieldSetsUS_1_EarthEngine.csv', 'wb') as out_file:
    out_file.write(r.content)

# Remove degenerate parts of polygons (i.e. linestrings) to create final map product files

In [ ]:
def get_final_geometry(x):
    x = shapely.from_geojson(x)
    if x.geom_type == 'GeometryCollection':
        polygons = list(polygonize(x.geoms))
        if len(polygons) == 1:
            return polygons[0]
        elif len(polygons) > 1:
            return MultiPolygon(polygons)
        else:
            raise ValueError("invalid geometry")
    else:
        return x

In [ ]:
i = 0
output_filename = 'inference_data/outputs/FieldSetsUS_1_FINAL.csv'
for df in tqdm(pd.read_csv("inference_data/outputs/FieldSetsUS_1_EarthEngine.csv", chunksize=1000000)):
    header = (i == 0)
    df.rename(columns={'.geo':'geometry'}, inplace=True)
    df['geometry'] = df['geometry'].apply(get_final_geometry)
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1

In [ ]:
i = 0
output_filename = 'inference_data/outputs/FieldSetsUS_2_FINAL.csv'
for df in tqdm(pd.read_csv("inference_data/outputs/FieldSetsUS_2_EarthEngine.csv", chunksize=1000000)):
    header = (i == 0)
    df.rename(columns={'.geo':'geometry'}, inplace=True)
    df['geometry'] = df['geometry'].apply(get_final_geometry)
    df.to_csv(output_filename, mode='a', index=False, header=header)
    i += 1